# Індивідуальні завдання до практичної роботи № 3


In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PATH = '/content/drive/MyDrive/sales.csv'

# Якщо файлу sales.csv немає на Диску - створюємо навчальний набір про продажі кави
import os
if not os.path.exists(PATH):
    rng = np.random.default_rng(42)
    n = 300
    drinks = {"Latte": 38.7, "Americano": 28.9, "Cappuccino": 32.0, "Espresso": 25.0, "Hot Chocolate": 35.8, "Cocoa": 35.8}
    names = rng.choice(list(drinks), n)
    cash = rng.choice(["card", "cash"], n, p=[0.9, 0.1])
    dt = pd.to_datetime("2024-03-01 08:00") + pd.to_timedelta(np.sort(rng.integers(0, 60*24*60, n)), unit="m")
    pd.DataFrame({
        "date": dt.date,
        "datetime": dt,
        "cash_type": cash,
        "card": [f"ANON-0000-0000-{i:04d}" if c == "card" else np.nan for i, c in enumerate(cash, 1)],
        "money": [round(drinks[x] * rng.choice([1.0, 1.0, 1.0, 1.1]), 2) for x in names],
        "coffee_name": names,
    }).to_csv(PATH, index=False)
    print("Створено навчальний файл:", PATH)

df = pd.read_csv(PATH)
df.head()

,date,datetime,cash_type,card,money,coffee_name
0,2024-03-01,2024-03-01 08:00:00,card,ANON-0000-0000-0000,24.0,Espresso
1,2024-03-01,2024-03-01 08:47:00,cash,ANON-0000-0000-0001,35.0,Cocoa
2,2024-03-01,2024-03-01 09:34:00,cash,ANON-0000-0000-0002,32.0,Cappuccino
3,2024-03-01,2024-03-01 10:21:00,card,ANON-0000-0000-0003,35.0,Cocoa
4,2024-03-01,2024-03-01 11:08:00,card,ANON-0000-0000-0004,35.0,Cocoa


## Завдання 1. «Брудний» DataFrame: пропуски, дублікати, викиди


In [4]:
dirty = pd.DataFrame({
    'id':     [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 5, 8],
    'name':   ['Anna', 'Bohdan', 'Olena', None, 'Petro', 'Iryna', 'Taras', 'Maria', 'Oleh', 'Sofia', 'Andrii', 'Petro', 'Maria'],
    'city':   ['Kyiv', 'Lviv', 'Kyiv', 'Odesa', None, 'Lviv', 'Kyiv', 'Lviv', 'Odesa', 'Kyiv', 'Lviv', None, 'Lviv'],
    'age':    [20, 21, None, 22, 24, 23, 21, 22, 25, 20, 23, 24, 22],
    'salary': [30000, 32000, 31000, 29000, 33000, None, 30500, 31500, 32500, 29500, 950000, 33000, 31500],
    'score':  [85.5, 90.0, 78.3, 88.0, 92.1, 81.0, None, 79.5, 86.0, 84.5, 87.0, 92.1, 79.5],
})
dirty

,id,name,city,age,salary,score
0,1,Anna,Kyiv,20.0,30000.0,85.5
1,2,Bohdan,Lviv,21.0,32000.0,90.0
2,3,Olena,Kyiv,NaN,31000.0,78.3
3,4,None,Odesa,22.0,29000.0,88.0
4,5,Petro,None,24.0,33000.0,92.1
5,6,Iryna,Lviv,23.0,NaN,81.0
6,7,Taras,Kyiv,21.0,30500.0,NaN
7,8,Maria,Lviv,22.0,31500.0,79.5
8,9,Oleh,Odesa,25.0,32500.0,86.0
9,10,Sofia,Kyiv,20.0,29500.0,84.5


In [5]:
print("Пропуски у кожній колонці:")
print(dirty.isnull().sum())
print("\nПовних дублікатів:", dirty.duplicated().sum())

Пропуски у кожній колонці:
id        0
name      1
city      2
age       1
salary    1
score     1
dtype: int64

Повних дублікатів: 2


In [6]:
clean1 = dirty.copy()

# 1) Пропуски: числові - медіаною, категоріальні - модою
for col in ['age', 'salary', 'score']:
    clean1[col] = clean1[col].fillna(clean1[col].median())
for col in ['name', 'city']:
    clean1[col] = clean1[col].fillna(clean1[col].mode()[0])

# 2) Повні дублікати
clean1 = clean1.drop_duplicates()
print("Після видалення дублікатів:", clean1.shape)

# 3) Викиди за IQR (колонка salary)
Q1, Q3 = clean1['salary'].quantile(0.25), clean1['salary'].quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
print(f"Межі IQR для salary: [{lower}, {upper}]")
clean1 = clean1[(clean1['salary'] >= lower) & (clean1['salary'] <= upper)]

print("Фінальний розмір:", clean1.shape)
clean1

Після видалення дублікатів: (11, 6)
Межі IQR для salary: [27250.0, 35250.0]
Фінальний розмір: (10, 6)


,id,name,city,age,salary,score
0,1,Anna,Kyiv,20.0,30000.0,85.50
1,2,Bohdan,Lviv,21.0,32000.0,90.00
2,3,Olena,Kyiv,22.0,31000.0,78.30
3,4,Maria,Odesa,22.0,29000.0,88.00
4,5,Petro,Lviv,24.0,33000.0,92.10
5,6,Iryna,Lviv,23.0,31500.0,81.00
6,7,Taras,Kyiv,21.0,30500.0,85.75
7,8,Maria,Lviv,22.0,31500.0,79.50
8,9,Oleh,Odesa,25.0,32500.0,86.00
9,10,Sofia,Kyiv,20.0,29500.0,84.50


## Завдання 2. `sales.csv`: додаємо пропуски та дублікати, потім очищуємо

In [7]:
df2 = df.copy()

# Штучно додаємо 5 пропусків у money
rng = np.random.default_rng(0)
nan_idx = rng.choice(df2.index, size=5, replace=False)
df2.loc[nan_idx, 'money'] = np.nan

# Штучно додаємо 3 дублікати (копії випадкових рядків)
dups = df2.sample(3, random_state=1)
df2 = pd.concat([df2, dups], ignore_index=True)

print("Пропусків у money:", df2['money'].isnull().sum())
print("Дублікатів:", df2.duplicated().sum())
print("Розмір:", df2.shape)

Пропусків у money: 6
Дублікатів: 3
Розмір: (303, 6)


In [8]:
# Очищення
df2['money'] = df2['money'].fillna(df2['money'].mean())          # пропуски -> середнє
df2 = df2.drop_duplicates()                                       # видалення дублікатів
df2['datetime'] = pd.to_datetime(df2['datetime'], errors='coerce')  # текст -> datetime64

print("Пропусків у money:", df2['money'].isnull().sum())
print("Дублікатів:", df2.duplicated().sum())
print("Розмір:", df2.shape)
print(df2.dtypes)

df2.to_csv('/content/drive/MyDrive/sales_cleaned_task2.csv', index=False)
print("Збережено!")

Пропусків у money: 0
Дублікатів: 0
Розмір: (300, 6)
date                   object
datetime       datetime64[ns]
cash_type              object
card                   object
money                 float64
coffee_name            object
dtype: object
Збережено!


## Завдання 3. Компоненти дати

In [9]:
df3 = df.copy()
df3['datetime'] = pd.to_datetime(df3['datetime'], errors='coerce')
df3['year'] = df3['datetime'].dt.year
df3['month'] = df3['datetime'].dt.month
df3['day'] = df3['datetime'].dt.day
df3['day_of_week'] = df3['datetime'].dt.dayofweek
df3['hour'] = df3['datetime'].dt.hour
df3[['datetime', 'year', 'month', 'day', 'day_of_week', 'hour']].head(10)

,datetime,year,month,day,day_of_week,hour
0,2024-03-01 08:00:00,2024,3,1,4,8
1,2024-03-01 08:47:00,2024,3,1,4,8
2,2024-03-01 09:34:00,2024,3,1,4,9
3,2024-03-01 10:21:00,2024,3,1,4,10
4,2024-03-01 11:08:00,2024,3,1,4,11
5,2024-03-01 11:55:00,2024,3,1,4,11
6,2024-03-01 12:42:00,2024,3,1,4,12
7,2024-03-01 13:29:00,2024,3,1,4,13
8,2024-03-01 14:16:00,2024,3,1,4,14
9,2024-03-01 15:03:00,2024,3,1,4,15


## Завдання 4. Обробка рядкових даних

In [10]:
cities = pd.DataFrame({
    'city': ['  Kyiv ', 'LVIV', ' Odesa', 'kharkiv  ', ' DNIPRO '],
    'phone': ['+38 (044) 111-22-33', '+38 (032) 222-33-44', '+38 (048) 333-44-55',
              '+38 (057) 444-55-66', '+38 (056) 555-66-77']
})
print("До очищення:")
print(cities)

cities['city'] = cities['city'].str.strip().str.lower()
cities['phone'] = cities['phone'].str.replace(r'[\s()+-]', '', regex=True)

print("\nПісля очищення:")
print(cities)

До очищення:
        city                phone
0      Kyiv   +38 (044) 111-22-33
1       LVIV  +38 (032) 222-33-44
2      Odesa  +38 (048) 333-44-55
3  kharkiv    +38 (057) 444-55-66
4    DNIPRO   +38 (056) 555-66-77

Після очищення:
      city         phone
0     kyiv  380441112233
1     lviv  380322223344
2    odesa  380483334455
3  kharkiv  380574445566
4   dnipro  380565556677


## Завдання 5. One-hot encoding колонки `cash_type`

In [11]:
df5 = pd.get_dummies(df, columns=['cash_type'])
df5.head()

,date,datetime,card,money,coffee_name,cash_type_card,cash_type_cash
0,2024-03-01,2024-03-01 08:00:00,ANON-0000-0000-0000,24.0,Espresso,True,False
1,2024-03-01,2024-03-01 08:47:00,ANON-0000-0000-0001,35.0,Cocoa,False,True
2,2024-03-01,2024-03-01 09:34:00,ANON-0000-0000-0002,32.0,Cappuccino,False,True
3,2024-03-01,2024-03-01 10:21:00,ANON-0000-0000-0003,35.0,Cocoa,True,False
4,2024-03-01,2024-03-01 11:08:00,ANON-0000-0000-0004,35.0,Cocoa,True,False


## Завдання 6. Нормалізація та стандартизація `money`

In [12]:
df6 = df.copy()
df6['money_norm'] = (df6['money'] - df6['money'].min()) / (df6['money'].max() - df6['money'].min())
df6['money_std'] = (df6['money'] - df6['money'].mean()) / df6['money'].std()

print("money_norm: min =", df6['money_norm'].min(), ", max =", df6['money_norm'].max())
print("money_std:  mean =", df6['money_std'].mean(), ", std =", df6['money_std'].std())

money_norm: min = 0.0 , max = 1.0
money_std:  mean = -2.1908401019269756e-16 , std = 0.9999999999999989


## Завдання 7. Ознака `is_card` через `np.where()`

In [13]:
df7 = df.copy()
df7['is_card'] = np.where(df7['cash_type'] == 'card', 1, 0)
df7[['cash_type', 'is_card']].head(10)

,cash_type,is_card
0,card,1
1,cash,0
2,cash,0
3,card,1
4,card,1
5,card,1
6,card,1
7,cash,0
8,card,1
9,card,1


## ★ Завдання 8. Функція `clean_dataset(df)`


In [14]:
def clean_dataset(frame, date_col=None):
    """Повертає очищену копію DataFrame."""
    out = frame.copy()

    # 1) повні дублікати
    out = out.drop_duplicates()

    # 2) колонка з датою -> datetime64
    if date_col is not None and date_col in out.columns:
        out[date_col] = pd.to_datetime(out[date_col], errors='coerce')

    # 3) пропуски: числові - медіана, категоріальні - мода
    num_cols = out.select_dtypes(include='number').columns
    cat_cols = [c for c in out.columns if c not in num_cols and c != date_col]
    for col in num_cols:
        out[col] = out[col].fillna(out[col].median())
    for col in cat_cols:
        if out[col].notna().any():
            out[col] = out[col].fillna(out[col].mode()[0])

    # 4) викиди (IQR) за всіма числовими колонками
    mask = pd.Series(True, index=out.index)
    for col in num_cols:
        Q1, Q3 = out[col].quantile(0.25), out[col].quantile(0.75)
        IQR = Q3 - Q1
        mask &= out[col].between(Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)

    return out[mask].reset_index(drop=True)

In [15]:
test = pd.DataFrame({
    'order_id': list(range(1, 16)) + [3, 7],
    'product': ['Latte', 'Latte', 'Mocha', None, 'Latte', 'Mocha', 'Cocoa', 'Latte', 'Mocha', 'Latte',
                'Cocoa', 'Latte', 'Mocha', 'Latte', 'Cocoa', 'Mocha', 'Cocoa'],
    'price':    [38, 38, 40, 38, None, 40, 36, 38, 40, 38, 36, 38, 40, 900, 36, 40, 36],
    'quantity': [1, 2, 1, 3, 2, None, 1, 2, 1, 1, 2, 1, 2, 1, 1, 1, 1],
    'order_date': ['2024-03-01', '2024-03-01', '2024-03-02', '2024-03-02', '2024-03-03', '2024-03-03',
                   '2024-03-04', '2024-03-04', '2024-03-05', '2024-03-05', '2024-03-06', '2024-03-06',
                   '2024-03-07', '2024-03-07', '2024-03-08', '2024-03-02', '2024-03-04'],
})
print("До очищення:", test.shape, "| пропусків:", test.isnull().sum().sum(), "| дублікатів:", test.duplicated().sum())

cleaned = clean_dataset(test, date_col='order_date')
print("Після очищення:", cleaned.shape, "| пропусків:", cleaned.isnull().sum().sum(), "| дублікатів:", cleaned.duplicated().sum())
print(cleaned.dtypes)
cleaned

До очищення: (17, 5) | пропусків: 3 | дублікатів: 2
Після очищення: (14, 5) | пропусків: 0 | дублікатів: 0
order_id               int64
product               object
price                float64
quantity             float64
order_date    datetime64[ns]
dtype: object


,order_id,product,price,quantity,order_date
0,1,Latte,38.0,1.0,2024-03-01
1,2,Latte,38.0,2.0,2024-03-01
2,3,Mocha,40.0,1.0,2024-03-02
3,4,Latte,38.0,3.0,2024-03-02
4,5,Latte,38.0,2.0,2024-03-03
5,6,Mocha,40.0,1.0,2024-03-03
6,7,Cocoa,36.0,1.0,2024-03-04
7,8,Latte,38.0,2.0,2024-03-04
8,9,Mocha,40.0,1.0,2024-03-05
9,10,Latte,38.0,1.0,2024-03-05


In [16]:
cleaned.to_csv('/content/drive/MyDrive/cleaned_test_dataset.csv', index=False)
print("Результат збережено!")

Результат збережено!
